# 2026-10-21_4 BC / Cone / AC 基因表达堆叠小提琴图（sharedY：同一基因胚胎/成年 y 轴一致）

**需求与口径**（沿用 2026-10-21_3 MG 版，颜色与风格完全一致）

- **基因 × 细胞类型**（用户指定；ZIC1 在三种细胞类型各画一次）：
  - **BC**：NSG1, BHLHE22, PDE10A, PODXL, ZIC1
  - **Cone**：NOL4, ZIC1, PLXDC1
  - **AC**：ZIC1
- **相同细胞类型的基因绘制到同一页**（堆叠小提琴，首个基因在顶部）；**不同细胞类型分页**；胚胎 / 成年分页（与 _1/_3 一致，跨期数值不直接比较）
- 细胞范围：仅 BC / Cone / AC；成年剔除 70years（Old_unpaired，仅 macula、无 peripheral 对照）
- 分组：stage × region（fovea/macula → central，peripheral → peripheral）；X 轴 Early-central; Early-peripheral; Mid-central; Mid-peripheral; Late-central; Late-peripheral（成年页 young/old 同序）
- 配色与 _3 一致：stage 本色 = central、浅色（lighten 45%）= peripheral；Early #59A14F / Mid #EDC948 / Late #B07AA1 / young #4E79A7 / old #E15759
- sharedY：**同一细胞类型同一基因**在胚胎页与成年页 y 轴一致（该基因在该细胞类型跨两期全局最大值 + geom_blank）
- 仅输出 PDF（11×7 英寸，6 页：BC-胚胎/成年、Cone-胚胎/成年、AC-胚胎/成年）

**数据源（只读，不修改原始对象）**

- 胚胎：`/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds`
- 成年：`/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds`

**输出**

- `Data/2026-10-21_4_BC_Cone_AC_gene_percell_expr.rds` 与 `.csv`（BC/Cone/AC 每细胞 counts + lognorm 表，7 基因）
- `Plot/2026-10-21_4_BC_Cone_AC_gene_expression_stacked_violin_sharedY.pdf`（6 页）

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(scales)
})
set.seed(1)

DATE_STR <- "2026-10-21_4"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")

# 原始 Seurat 对象（只读）
EMB_RDS <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds"
ADU_RDS <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds"

# 基因 × 细胞类型（用户指定顺序 = 图上自上而下；ZIC1 三种细胞类型各画一次）
GENES_BY_CT <- list(
  BC   = c("NSG1", "BHLHE22", "PDE10A", "PODXL", "ZIC1"),
  Cone = c("NOL4", "ZIC1", "PLXDC1"),
  AC   = c("ZIC1")
)
GENES_ALL <- unique(unlist(GENES_BY_CT))
CELLTYPES <- names(GENES_BY_CT)
cat("细胞类型:", paste(CELLTYPES, collapse = " / "), "\n")
for (ct in CELLTYPES) cat(sprintf("  %-4s: %s\n", ct, paste(GENES_BY_CT[[ct]], collapse = ", ")))
cat("全部基因(去重):", paste(GENES_ALL, collapse = ", "), "\n")

STAGE_MAP_EMB <- c(
  "70days" = "Early",  "79days" = "Early",  "87days" = "Early",  "91days" = "Early",
  "100days" = "Mid",   "103days" = "Mid",   "116days" = "Mid",
  "137days" = "Late",  "141days" = "Late",  "142days" = "Late",
  "162days" = "Late",  "165days" = "Late"
)
STAGE_MAP_ADU <- c(
  "21years" = "young", "25years" = "young", "30years" = "young", "32years" = "young",
  "67years" = "old",   "75years" = "old",   "86years" = "old"
)

GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

# ---- 配色（与 2026-10-21_3 完全一致）-----------------------------------------
cols.stage.full <- c('#59A14F','#EDC948','#B07AA1','#4E79A7','#E15759')
names(cols.stage.full) <- c('Early','Mid','Late','Young','Old')
cols.stage      <- cols.stage.full[c("Early","Mid","Late")]
cols.adultstage <- setNames(cols.stage.full[c("Young","Old")], c("young","old"))
lighten_col <- function(col, f = 0.45) {
  m <- grDevices::col2rgb(col)
  grDevices::rgb(t(m + (255 - m) * f), maxColorValue = 255)
}
region_cols <- function(base) {
  v <- c(base, sapply(base, lighten_col))
  names(v) <- c(paste0(names(base), "_central"), paste0(names(base), "_peripheral"))
  v
}
cols.emb.groups <- region_cols(cols.stage)
names(cols.emb.groups) <- gsub("_", "-", names(cols.emb.groups))
cols.adu.groups <- region_cols(cols.adultstage)
names(cols.adu.groups) <- gsub("_", "-", names(cols.adu.groups))

stopifnot(file.exists(EMB_RDS), file.exists(ADU_RDS))
cat("\n胚胎分组配色：\n"); print(cols.emb.groups[GROUP_LEVELS_EMB])
cat("\n成年分组配色：\n"); print(cols.adu.groups[GROUP_LEVELS_ADU])
cat("\nR", as.character(getRversion()), "| Seurat", as.character(packageVersion("Seurat")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")

In [ ]:
# ---- 2. 提取函数：读入对象 -> BC/Cone/AC -> 抽取 7 基因 counts + lognorm ----
# 与 2026-10-21_1 的 extract_mg_table 同口径，仅细胞类型范围不同
extract_ct_table <- function(rds_path, period, stage_map, group_levels, celltypes, genes) {
  obj <- readRDS(rds_path)
  stopifnot(all(c("celltype", "tissue", "donor_age", "nCount_RNA") %in% colnames(CONTACT_EMAIL)))
  stopifnot(all(genes %in% rownames(obj)))
  cat(sprintf("[%s] 对象细胞数 %d | 基因数 %d\n", period, ncol(obj), nrow(obj)))

  meta <- CONTACT_EMAIL
  keep <- which(meta$celltype %in% celltypes)
  cells <- rownames(meta)[keep]
  m <- meta[keep, , drop = FALSE]
  cat(sprintf("[%s] %s 细胞数 %d\n", period, paste(celltypes, collapse = "/"), length(cells)))
  print(table(m$celltype))

  tissue_l <- tolower(as.character(m$tissue))
  region <- ifelse(tissue_l %in% c("fovea", "macula", "central"), "central",
                   ifelse(tissue_l == "peripheral", "peripheral", NA_character_))
  stage <- unname(stage_map[as.character(m$donor_age)])

  excl <- is.na(stage) | is.na(region)
  if (any(excl)) {
    cat(sprintf("[%s] 剔除无 stage/region 归属细胞 %d 个（donor_age: %s）\n",
                period, sum(excl),
                paste(sort(unique(as.character(m$donor_age[excl]))), collapse = ",")))
  }
  m <- m[!excl, , drop = FALSE]
  region <- region[!excl]
  stage <- stage[!excl]
  cells <- cells[!excl]

  counts_mat <- GetAssayData(obj, assay = "RNA", layer = "counts")[genes, cells]
  data_mat   <- GetAssayData(obj, assay = "RNA", layer = "data")[genes, cells]
  stopifnot(identical(colnames(counts_mat), cells))

  counts_df  <- as.data.frame(t(as.matrix(counts_mat)))
  lognorm_df <- as.data.frame(t(as.matrix(data_mat)))
  colnames(counts_df)  <- paste0(genes, "_counts")
  colnames(lognorm_df) <- paste0(genes, "_lognorm")

  out <- data.frame(
    cell       = cells,
    period     = period,
    celltype   = as.character(m$celltype),
    donor_age  = as.character(m$donor_age),
    tissue     = as.character(m$tissue),
    region     = region,
    stage      = stage,
    nCount_RNA = as.numeric(m$nCount_RNA),
    stringsAsFactors = FALSE
  )
  out$stage_region <- factor(paste(out$stage, out$region, sep = "-"), levels = group_levels)
  out <- cbind(out, counts_df, lognorm_df)
  rownames(out) <- NULL
  stopifnot(!any(is.na(out$stage_region)), !any(is.na(out$nCount_RNA)))

  rm(obj, meta, m, counts_mat, data_mat, counts_df, lognorm_df)
  invisible(gc(verbose = FALSE))
  out
}
cat("extract_ct_table() 定义完成\n")

In [ ]:
# ---- 3. 胚胎数据提取 --------------------------------------------------------
emb <- extract_ct_table(EMB_RDS, "Embryonic", STAGE_MAP_EMB, GROUP_LEVELS_EMB, CELLTYPES, GENES_ALL)

cat("\n[Embryonic] celltype × stage_region 细胞数：\n")
print(table(emb$celltype, droplevels(factor(as.character(emb$stage_region), levels = GROUP_LEVELS_EMB))))
cat("\n[Embryonic] 各基因检出率（counts > 0 细胞占比）：\n")
print(round(colMeans(emb[paste0(GENES_ALL, "_counts")] > 0), 4))

In [ ]:
# ---- 4. 成年数据提取（剔除 70years）-----------------------------------------
adu <- extract_ct_table(ADU_RDS, "Adult", STAGE_MAP_ADU, GROUP_LEVELS_ADU, CELLTYPES, GENES_ALL)
stopifnot(!("70years" %in% adu$donor_age))
cat("已确认 70years 已剔除\n")

cat("\n[Adult] celltype × stage_region 细胞数：\n")
print(table(adu$celltype, droplevels(factor(as.character(adu$stage_region), levels = GROUP_LEVELS_ADU))))
cat("\n[Adult] 各基因检出率（counts > 0 细胞占比）：\n")
print(round(colMeans(adu[paste0(GENES_ALL, "_counts")] > 0), 4))

In [ ]:
# ---- 5. 合并两期并保存每细胞表达表 -------------------------------------------
percell <- rbind(emb, adu)
percell$period <- factor(percell$period, levels = c("Embryonic", "Adult"))

cat("合并后总细胞数:", nrow(percell), "\n")
print(table(percell$period, percell$celltype))

percell_rds <- file.path(DATA_DIR, paste0(DATE_STR, "_BC_Cone_AC_gene_percell_expr.rds"))
percell_csv <- file.path(DATA_DIR, paste0(DATE_STR, "_BC_Cone_AC_gene_percell_expr.csv"))
saveRDS(percell, percell_rds)
write.csv(percell, percell_csv, row.names = FALSE, quote = FALSE)
cat("已保存:", basename(percell_rds), "/", basename(percell_csv), "\n")